# 🚀 P3 · Training in Google Colab

<a href="https://colab.research.google.com/github/rdf-sul/YOLO_Training/blob/main/projekt-vorlage/P3_training_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

**Projekt-Notebook 3 von 3** · läuft in **Google Colab** mit kostenloser GPU

```
   P1 Daten sammeln  →  P2 Labeln & prüfen  →  ▶ P3 Training in Colab
                                               ─────────────────────
               Training · Fortsetzen · Auswerten · Sichern · Vergleichen · Abschlusstest
```

### Das Prinzip: Eine Runde = ein Durchlauf des ganzen Notebooks

| Runde | Was du oben in den ⚙️ Einstellungen änderst |
|---|---|
| **1** (erstes Training) | `DATENSATZ = "datensatz_v1"`, `LAUF = "v1_yolo26n"`, `MODELL = "yolo26n.pt"` |
| **2** (Nachtraining) | `DATENSATZ = "datensatz_v2"`, `LAUF = "v2_feintuning"`, `MODELL = "modelle/v1_yolo26n_best.pt"`, `VERGLEICH_MIT = "v1_yolo26n"` |
| **Ende** | `ABSCHLUSSTEST = True` beim Lauf, für den du dich entschieden hast |

Danach: Menü **Laufzeit → Alle ausführen**. Jede Runde bekommt so automatisch Lernkurven, Kennzahlen,
Fehleranalyse, besten Schwellenwert und eine gesicherte Modelldatei.

| Abschnitt | Was passiert |
|---|---|
| ⚙️ Einstellungen | Datensatz, Laufname, Startmodell, Epochen … |
| 1–3 · Vorbereitung | GPU prüfen, Ultralytics installieren, Google Drive verbinden |
| 4 · Datensatz | ZIP aus Drive holen und prüfen |
| 5 · Training | starten **oder nach Abbruch automatisch fortsetzen** |
| 6 · Auswerten | Lernkurven, Kennzahlen pro Klasse, Confusion Matrix, Schwellenwert, **Fehleranalyse** |
| 7 · Sichern | Modell und Infodatei in Drive, Download |
| 8 · Ausprobieren | eigene Fotos erkennen lassen |
| 9 · Vergleich | nur beim Nachtraining: neues gegen altes Modell, fair auf denselben Bildern |
| 10 · Abschlusstest | nur ganz am Ende: einmalige Bewertung mit den Testbildern |

> 📖 Theorie: [P3 auf der Lernseite](https://rdf-sul.github.io/YOLO_Training/projekt/p3-training/) · [Ergebnisse verstehen](https://rdf-sul.github.io/YOLO_Training/training/ergebnisse/) ·
> [Nachtraining](https://rdf-sul.github.io/YOLO_Training/projekt/nachtraining/)

### Vorbereitung (Pflicht!)

1. **Kopie speichern:** Menü **Datei → Kopie in Drive speichern**. Arbeite ab jetzt **nur** in dieser Kopie.
   Sonst sind deine Einstellungen nach einem Abbruch weg.
2. **GPU einschalten:** Menü **Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern**.
3. Das Datensatz-ZIP aus P2 liegt in Google Drive unter **`Meine Ablage/YOLO_Projekt/datensaetze/`**.

### 🛟 Was passiert, wenn Colab abbricht?

Das Training speichert nach **jeder Epoche** einen Zwischenstand (`last.pt`) direkt in deinem **Google Drive**.
Nach einem Abbruch öffnest du deine Kopie des Notebooks erneut und wählst **Laufzeit → Alle ausführen**.
Das Training läuft an der unterbrochenen Stelle weiter. **Wichtig:** Die Einstellungen dürfen sich dazwischen
nicht ändern, vor allem nicht `LAUF` und `DATENSATZ`.

---
## ⚙️ Einstellungen

**Nur hier** änderst du etwas. Ein **Lauf** ist ein Trainingsvorgang mit festen Einstellungen. Er bekommt einen Namen.

| Einstellung | Bedeutung |
|---|---|
| `DATENSATZ` | Name der ZIP-Datei aus P2 (ohne `.zip`) |
| `LAUF` | Name dieses Trainings. **Jeder neue Versuch bekommt einen neuen Namen!** Gleicher Name = fortsetzen. |
| `MODELL` | Startmodell: `yolo26n.pt` (schnell) … `yolo26s.pt` / `yolo26m.pt` (genauer, langsamer) oder ein eigenes Modell aus Drive, z. B. `modelle/v1_yolo26n_best.pt` |
| `VERGLEICH_MIT` | nur beim Nachtraining: Name des vorherigen Laufs, sonst `""` |
| `EPOCHEN` | Höchstzahl der Durchgänge durch alle Trainingsbilder (Feintuning: etwa 60 % davon) |
| `PATIENCE` | Stopp, wenn sich mAP50-95 so viele Epochen lang nicht verbessert (*Early Stopping*) |
| `BILDGROESSE` | Bilder werden auf diese Größe skaliert: 640 Standard, 1280 bei sehr kleinen Objekten (dann `BATCH = 8`) |
| `BATCH` | Bilder pro Rechenschritt. Bei „CUDA out of memory“ halbieren |
| `ZUSATZ` | weitere Trainingsparameter, siehe Kasten unten |
| `ABSCHLUSSTEST` | erst auf `True`, wenn du dich für dein **endgültiges** Modell entschieden hast |

In [ ]:
PROJEKT_ORDNER = "YOLO_Projekt"     # Ordner in „Meine Ablage“ in Google Drive
DATENSATZ      = "datensatz_v1"     # ZIP aus P2 (ohne .zip)
LAUF           = "v1_yolo26n"       # Name dieses Trainingslaufs
MODELL         = "yolo26n.pt"       # Startmodell (vortrainiert auf COCO → Transfer Learning)
VERGLEICH_MIT  = ""                 # Nachtraining: vorheriger Lauf, z. B. "v1_yolo26n"
EPOCHEN        = 100
PATIENCE       = 30
BILDGROESSE    = 640
BATCH          = 16
ZUSATZ         = {}                 # z. B. {"flipud": 0.5, "degrees": 90} bei Fotos von oben
ABSCHLUSSTEST  = False              # erst ganz am Ende auf True setzen!

💡 **`ZUSATZ` – nur bei Bedarf.** Ultralytics verändert die Trainingsbilder zufällig (*Augmentation*), damit das Modell
robuster wird. Standardmäßig wird nur **links/rechts** gespiegelt.

| Situation | `ZUSATZ` |
|---|---|
| Fotos **von oben** (Bauteile auf dem Tisch), Lage egal | `{"flipud": 0.5, "degrees": 90}` (auch oben/unten spiegeln, drehen) |
| Links/rechts ist wichtig (z. B. Pfeile, Schrift) | `{"fliplr": 0.0}` |
| Sehr wenige Bilder (< 100) beim Nachtraining | `{"freeze": 10}` (die ersten 10 Schichten bleiben fest) |

---
## 1 · GPU prüfen

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules

try:
    import torch
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
    else:
        print("⚠️  Keine GPU! Training auf der CPU dauert sehr lange.")
        print("   → Laufzeit → Laufzeittyp ändern → T4 GPU → Speichern, danach Laufzeit → Alle ausführen.")
except ImportError:
    print("PyTorch fehlt – wird mit Ultralytics installiert.")

**Keine GPU verfügbar?** Das kostenlose GPU-Kontingent ist pro Google-Konto begrenzt und erholt sich nach einigen Stunden.
Auswege: später erneut versuchen · mit dem Konto einer anderen Person aus der Gruppe trainieren (ZIP dort hochladen) ·
notfalls auf der CPU mit `BILDGROESSE = 416` und `EPOCHEN = 30` (dauert trotzdem lange).

---
## 2 · Ultralytics installieren

In Colab nutzen wir **uv**: Es installiert deutlich schneller als pip. `--system` installiert direkt in das Python von Colab.
Die Version ist **festgelegt** (8.4.214), damit ein fortgesetztes Training in jeder Sitzung mit derselben Version läuft.

In [ ]:
if IN_COLAB:
    subprocess.run("pip install -q uv && uv pip install --system -q ultralytics==8.4.214", shell=True, check=True)

import ultralytics
from ultralytics import YOLO
print("✅ Ultralytics", ultralytics.__version__)

---
## 3 · Google Drive verbinden

Es erscheint ein Fenster zur Anmeldung: Google-Konto wählen und den Zugriff erlauben.
Alle Trainingsergebnisse werden dann **direkt in Drive** gespeichert und überstehen so jeden Abbruch.

In [ ]:
from pathlib import Path
import shutil, json, zipfile

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASIS = Path("/content/drive/MyDrive") / PROJEKT_ORDNER
    DATEN_LOKAL = Path("/content/daten")      # schnelle lokale Festplatte der Colab-Sitzung
else:  # zum Testen auf dem eigenen Rechner
    BASIS = (Path("drive_ersatz") / PROJEKT_ORDNER).resolve()   # absolute Pfade (sonst legt Ultralytics
    DATEN_LOKAL = Path("_colab_daten").resolve()                 #  relative Ordner unter runs/detect/ an)

DATENSAETZE, LAEUFE, MODELLE = BASIS / "datensaetze", BASIS / "laeufe", BASIS / "modelle"
for ordner in (DATENSAETZE, LAEUFE, MODELLE, DATEN_LOKAL):
    ordner.mkdir(parents=True, exist_ok=True)

print(f"📁 Projektordner: {BASIS}")
print("   Datensätze:", ", ".join(p.name for p in sorted(DATENSAETZE.glob("*.zip"))) or "– keine –")
print("   Trainingsläufe:", ", ".join(p.name for p in sorted(LAEUFE.iterdir()) if p.is_dir()) or "– noch keine –")
print("   Modelle:", ", ".join(p.name for p in sorted(MODELLE.glob("*.pt"))) or "– noch keine –")

---
## 4 · Datensatz holen und prüfen

Das ZIP wird aus Drive auf die **lokale Festplatte** der Colab-Sitzung kopiert und entpackt. Dort liest das Training
die Bilder viel schneller als direkt aus Drive. Wurde das ZIP in Drive ersetzt, wird automatisch neu entpackt.

In [ ]:
import pandas as pd

def datensatz_holen(name):
    """Kopiert <name>.zip aus Drive, entpackt es lokal und gibt den Pfad zur data.yaml zurück."""
    zip_drive = DATENSAETZE / f"{name}.zip"
    if not zip_drive.exists():
        raise FileNotFoundError(f"{zip_drive} fehlt – ZIP aus P2 in Drive hochladen (Name prüfen!)")
    ziel, merker = DATEN_LOKAL / name, DATEN_LOKAL / f"{name}.stand"
    stand = f"{zip_drive.stat().st_size}-{int(zip_drive.stat().st_mtime)}"
    if not (ziel / "data.yaml").exists() or not merker.exists() or merker.read_text() != stand:
        print(f"⏳ Entpacke {zip_drive.name} …")
        shutil.rmtree(ziel, ignore_errors=True)
        with zipfile.ZipFile(zip_drive) as z:
            z.extractall(DATEN_LOKAL)
        merker.write_text(stand)
    return (ziel / "data.yaml").resolve()

def info_zeigen(data_yaml):
    info = json.loads((Path(data_yaml).parent / "info.json").read_text(encoding="utf-8"))
    print(f"✅ {info['datensatz']} (erstellt {info['erstellt'][:10]}) – {info.get('notiz', '')}")
    print(f"   Klassen: {', '.join(info['klassen'])}\n")
    display(pd.DataFrame(info["anzahl"]).reindex(["bilder"] + info["klassen"]).fillna(0).astype(int))
    return info

DATA_YAML = datensatz_holen(DATENSATZ)
info = info_zeigen(DATA_YAML)

---
## 5 · Training – starten oder fortsetzen

Die Funktion `trainieren` schaut zuerst in Drive nach, ob es den Lauf `LAUF` schon gibt:

| Zustand in Drive | Was passiert |
|---|---|
| Lauf existiert noch nicht | **neues Training** ab dem Startmodell `MODELL` |
| Zwischenstand vorhanden, Training **unterbrochen** | **Fortsetzen** (`resume=True`) ab der letzten gespeicherten Epoche, mit allen Einstellungen von damals |
| Training **abgeschlossen** | nichts zu tun, es geht mit der Auswertung weiter |

⏱️ Mit T4-GPU dauert eine Epoche bei ~200 Bildern etwa 10–20 Sekunden. Lass den Browser-Tab **geöffnet**.
Während das Training läuft, kannst du auf der Lernseite [Ergebnisse verstehen](https://rdf-sul.github.io/YOLO_Training/training/ergebnisse/) lesen.

In [ ]:
import torch

def _stand_lesen(pfad):
    """Liest einen Zwischenstand. Gibt None zurück, wenn die Datei beschädigt ist (Abbruch beim Speichern)."""
    try:
        return torch.load(pfad, map_location="cpu", weights_only=False)
    except Exception:
        return None

def trainieren(lauf, modell, data_yaml, **parameter):
    """Startet ein neues Training – oder setzt ein unterbrochenes automatisch fort."""
    lauf_ordner = LAEUFE / lauf
    gewichte = lauf_ordner / "weights"
    if (gewichte / "last.pt").exists():
        datei, stand = gewichte / "last.pt", _stand_lesen(gewichte / "last.pt")
        if stand is None and (gewichte / "best.pt").exists():  # last.pt kaputt → mit best.pt weiter
            print("⚠️  last.pt ist beschädigt (Abbruch beim Speichern?) → setze mit best.pt fort.")
            datei, stand = gewichte / "best.pt", _stand_lesen(gewichte / "best.pt")
        if stand is None:
            raise RuntimeError(f"Der Zwischenstand in {gewichte} ist beschädigt. Neuen LAUF-Namen wählen und neu starten.")
        epoche, gesamt = stand.get("epoch", -1), stand.get("train_args", {}).get("epochs", 0)
        if epoche == -1 or epoche + 1 >= gesamt:
            print(f"✅ Lauf „{lauf}“ ist bereits fertig trainiert → weiter mit der Auswertung.")
            print("   (Für ein neues Training einen neuen LAUF-Namen wählen.)")
            return lauf_ordner
        print(f"🔁 Unterbrochenes Training gefunden: {epoche + 1} von {gesamt} Epochen fertig → wird fortgesetzt.")
        YOLO(str(datei)).train(resume=True)
    else:
        print(f"🚀 Neues Training „{lauf}“ mit {modell}")
        YOLO(str(modell)).train(data=str(data_yaml), project=str(LAEUFE), name=lauf, exist_ok=True, **parameter)
    return lauf_ordner

START = BASIS / MODELL if (BASIS / MODELL).exists() else MODELL    # eigenes Modell aus Drive oder yolo26n.pt
LAUF_ORDNER = trainieren(LAUF, START, DATA_YAML, epochs=EPOCHEN, patience=PATIENCE,
                         imgsz=BILDGROESSE, batch=BATCH, plots=True, **ZUSATZ)
BEST = LAUF_ORDNER / "weights" / "best.pt"
print(f"\n🏆 Bestes Modell: {BEST}")

---
## 6 · Auswerten

### 6.1 Lernkurven: Hat das Training funktioniert?

- **Loss** (Fehler) sollte bei **train** und **val** sinken.
- **mAP50-95** (die wichtigste Kennzahl) sollte steigen und sich dann einpendeln.
  Die **beste Epoche** (rote Linie) wird als `best.pt` gespeichert.
- **Overfitting** (Auswendiglernen) erkennst du daran, dass **mAP50-95 auf val** wieder **fällt** oder stagniert,
  während der train-Loss weiter sinkt. Ein steigender val-Loss allein ist noch kein sicheres Zeichen.
- **Normal:** In den **letzten 10 Epochen** fällt der train-Loss oft sprunghaft. Dann schaltet Ultralytics die
  Mosaik-Augmentation ab (`close_mosaic`), und die Trainingsbilder werden „leichter“.
- Endete das Training vor `EPOCHEN`? Dann hat **Early Stopping** gegriffen: `PATIENCE` Epochen lang keine Verbesserung.

In [ ]:
# @title Lernkurven zeichnen
import matplotlib.pyplot as plt

def lernkurven(lauf_ordner):
    df = pd.read_csv(lauf_ordner / "results.csv")
    df.columns = df.columns.str.strip()
    beste = df.loc[df["metrics/mAP50-95(B)"].idxmax(), "epoch"]
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    for name, farbe in (("box_loss", "#4363d8"), ("cls_loss", "#f58231")):
        for teil, stil in (("train", "-"), ("val", "--")):
            spalte = f"{teil}/{name}"
            if spalte in df:
                ax[0 if name == "box_loss" else 1].plot(df.epoch, df[spalte], stil, color=farbe, label=teil)
    ax[0].set_title("Box-Loss (Wie genau sitzen die Boxen?)")
    ax[1].set_title("Klassen-Loss (Wie sicher ist die Klasse?)")
    for spalte, farbe in (("metrics/mAP50(B)", "#3cb44b"), ("metrics/mAP50-95(B)", "#911eb4"),
                          ("metrics/precision(B)", "#aaa"), ("metrics/recall(B)", "#666")):
        ax[2].plot(df.epoch, df[spalte], color=farbe, label=spalte.split("/")[1].replace("(B)", ""))
    ax[2].set_title("Kennzahlen auf val")
    ax[2].set_ylim(0, 1)
    for a in ax:
        a.axvline(beste, color="red", lw=1)
        a.set_xlabel("Epoche")
        a.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    z = df.loc[df.epoch == beste].iloc[0]
    print(f"Beste Epoche: {int(beste)} von {int(df.epoch.max())}  ·  mAP50 = {z['metrics/mAP50(B)']:.3f}  ·  "
          f"mAP50-95 = {z['metrics/mAP50-95(B)']:.3f}")

lernkurven(LAUF_ORDNER)

✏️ **Ablesen:** In welcher Epoche lag das beste Modell? Hat Early Stopping gegriffen? Gibt es Anzeichen für Overfitting,
und woran erkennst du sie?

### 6.2 Kennzahlen pro Klasse (Validierungsbilder)

| Spalte | Frage | |
|---|---|---|
| **Precision (P)** | Wenn das Modell „Widerstand“ sagt, wie oft stimmt das? | wenige Fehlalarme |
| **Recall (R)** | Von allen echten Widerständen, wie viele findet es? | wenig übersehen |
| **mAP50** | Fläche unter der Precision-Recall-Kurve, Box gilt ab IoU ≥ 0,5 als Treffer | 0 … 1 |
| **mAP50-95** | wie mAP50, aber gemittelt über strengere Überlappungen (IoU 0,5 … 0,95) | **wichtigste Zahl** |

In [ ]:
modell = YOLO(str(BEST))
val = modell.val(data=str(DATA_YAML), split="val", imgsz=BILDGROESSE, plots=True,
                 project=str(DATEN_LOKAL / "auswertung"), name=f"{LAUF}_val", exist_ok=True, verbose=False)
VAL_ORDNER = Path(val.save_dir)

tab = pd.DataFrame(val.summary()).rename(columns={"Class": "Klasse", "Images": "Bilder", "Instances": "Objekte",
                                                  "Box-P": "Precision", "Box-R": "Recall", "Box-F1": "F1"})
tab = tab[["Klasse", "Bilder", "Objekte", "Precision", "Recall", "mAP50", "mAP50-95"]].round(3)
display(tab.style.background_gradient(subset=["mAP50-95"], cmap="RdYlGn", vmin=0, vmax=1))
print(f"Gesamt: Precision {val.box.mp:.3f} · Recall {val.box.mr:.3f} · mAP50 {val.box.map50:.3f} · "
      f"mAP50-95 {val.box.map:.3f}")
schwach = tab.sort_values("mAP50-95").iloc[0]
print(f"🔍 Schwächste Klasse: {schwach.Klasse} (mAP50-95 = {schwach['mAP50-95']:.3f})")

✏️ **Vergleiche** mit deiner Vermutung aus P2 (Abschnitt 8.1): Ist die schwächste Klasse die, die du erwartet hast?
Bei wenigen val-Bildern (unter ~10 pro Klasse) schwanken die Werte stark. Nimm sie dann nur als grobe Richtung.

### 6.3 Confusion Matrix: Was wird womit verwechselt?

Jede **Spalte** ist die **wahre** Klasse, jede **Zeile** die **Vorhersage** (normiert: Anteile je Spalte).
Ideal ist eine **dunkle Diagonale**. Die Zeile/Spalte **background** bedeutet:

- Spalte *background*, Zeile *Klasse X*: **Fehlalarm**. Das Modell sieht X, wo nichts ist.
- Zeile *background*, Spalte *Klasse X*: **übersehen**. Ein echtes X wurde nicht erkannt.

Ultralytics berechnet die Matrix mit einem festen Schwellenwert von 0,25. Die Zahlen können deshalb etwas von der
Fehleranalyse (6.5) abweichen, die den besten Schwellenwert aus 6.4 verwendet.

In [ ]:
from IPython.display import Image as Bild, display

def bild_zeigen(pfad, breite=750):
    if Path(pfad).exists():
        display(Bild(filename=str(pfad), width=breite))
    else:
        print("nicht gefunden:", pfad)

bild_zeigen(VAL_ORDNER / "confusion_matrix_normalized.png")

✏️ Welche Verwechslung ist am häufigsten? Würde auch eine Fachkraft diese beiden Klassen leicht verwechseln?

### 6.4 Der beste Schwellenwert (*Confidence*)

Jede Erkennung hat eine **Confidence** (0 … 1). Erst ab einem **Schwellenwert** zählt sie.

- **Niedriger** Schwellenwert: Das Modell findet mehr Objekte (Recall ↑), meldet aber auch mehr Fehlalarme (Precision ↓).
- **Hoher** Schwellenwert: weniger Fehlalarme, aber mehr Übersehenes.

Die **F1-Kurve** verbindet beides. Ihr Maximum ist ein guter Startwert für den Einsatz.

In [ ]:
f1 = val.box.f1_curve.mean(0)
BESTE_CONF = float(val.box.px[f1.argmax()])
plt.figure(figsize=(7, 3.5))
for i, k in enumerate(val.box.f1_curve):
    plt.plot(val.box.px, k, lw=1, alpha=.6, label=modell.names[int(val.box.ap_class_index[i])])
plt.plot(val.box.px, f1, "k", lw=2.5, label="alle Klassen")
plt.axvline(BESTE_CONF, color="red", ls="--")
plt.xlabel("Confidence-Schwellenwert"); plt.ylabel("F1"); plt.ylim(0, 1); plt.legend(fontsize=8)
plt.title(f"F1-Kurve – bester Schwellenwert ≈ {BESTE_CONF:.2f}")
plt.show()
print(f"✅ Empfohlener Schwellenwert: conf = {BESTE_CONF:.2f}  (F1 = {f1.max():.3f})")

✏️ **Überlege:** Ein Modell soll **fehlende Schutzbrillen** melden. Würdest du den Schwellenwert höher oder niedriger
als den F1-Wert wählen? Und bei einem Modell, das bei jeder Erkennung eine Maschine **stoppt**?

### 6.5 Fehleranalyse: **Welche** Fehler macht das Modell, und **wo**?

Kennzahlen sagen *wie gut*, die Fehleranalyse sagt *was falsch läuft*. Für jedes Validierungsbild werden die
Vorhersagen (mit dem besten Schwellenwert) mit deinen Labels verglichen. Treffer heißt: **IoU ≥ 0,5**, also
Schnittfläche ÷ Vereinigungsfläche der beiden Boxen mindestens 0,5.

| Farbe | Fehlerart | Ursache meistens … | Abhilfe beim Nachtraining |
|---|---|---|---|
| grün | **richtig** | – | – |
| violett | **verwechselt**: richtige Stelle, falsche Klasse | Klassen sehen sich ähnlich | mehr Bilder beider Klassen, Definition schärfen |
| rot | **Fehlalarm**: Box, wo kein Objekt ist | ähnliche Störer im Bild | **Hintergrundbilder** mit diesen Störern |
| orange | **übersehen**: echtes Objekt nicht gefunden | zu klein, verdeckt, ungewohnte Lage/Licht | mehr Bilder **genau dieser Situation** |

⚠️ Manchmal ist auch das **Label** falsch und das Modell hat recht! Dann in Label Studio korrigieren.

In [ ]:
# @title Fehleranalyse (Code ausgeblendet – Doppelklick zeigt ihn)
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
SCHRIFT = font_manager.findfont("DejaVu Sans")   # Schrift mit Umlauten

def _iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    schnitt = max(0, x2 - x1) * max(0, y2 - y1)
    return schnitt / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - schnitt + 1e-9)

def fehleranalyse(modell_pfad, data_yaml, teilmenge="val", conf=0.25, iou_grenze=0.5, zeigen=6):
    m = YOLO(str(modell_pfad))
    namen = m.names
    bilder = sorted((Path(data_yaml).parent / "images" / teilmenge).glob("*.jpg"))
    zaehler = {k: {"richtig": 0, "verwechselt": 0, "Fehlalarm": 0, "übersehen": 0} for k in namen.values()}
    pro_bild = []
    for r in m.predict([str(b) for b in bilder], conf=conf, imgsz=BILDGROESSE, stream=True, verbose=False):
        H, W = r.orig_shape
        txt = Path(data_yaml).parent / "labels" / teilmenge / (Path(r.path).stem + ".txt")
        wahr = []
        for z in (txt.read_text().split("\n") if txt.exists() else []):
            if z.strip():
                k, xc, yc, w, h = map(float, z.split())
                wahr.append((int(k), [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]))
        vorh = sorted(zip(r.boxes.cls.int().tolist(), r.boxes.xyxy.tolist(), r.boxes.conf.tolist()), key=lambda v: -v[2])
        benutzt, offen, markierungen = set(), [], []
        # Durchgang 1: Treffer mit gleicher Klasse (wie bei der mAP-Berechnung)
        for k, box, c in vorh:
            kand = [(_iou(box, wb), i) for i, (wk, wb) in enumerate(wahr) if i not in benutzt and wk == k]
            beste, i = max(kand, default=(0, None))
            if beste >= iou_grenze:
                benutzt.add(i)
                zaehler[namen[k]]["richtig"] += 1
                markierungen.append((box, "#2ca02c", namen[k]))
            else:
                offen.append((k, box))
        # Durchgang 2: übrige Vorhersagen an Stellen mit anderer Klasse = verwechselt, sonst Fehlalarm
        fehler = 0
        for k, box in offen:
            kand = [(_iou(box, wb), i) for i, (wk, wb) in enumerate(wahr) if i not in benutzt]
            beste, i = max(kand, default=(0, None))
            if beste >= iou_grenze:
                benutzt.add(i)
                zaehler[namen[wahr[i][0]]]["verwechselt"] += 1
                markierungen.append((box, "#9b30ff", f"{namen[wahr[i][0]]}→{namen[k]}"))
            else:
                zaehler[namen[k]]["Fehlalarm"] += 1
                markierungen.append((box, "#e6194b", f"Fehlalarm {namen[k]}"))
            fehler += 1
        for i, (wk, wb) in enumerate(wahr):
            if i not in benutzt:
                zaehler[namen[wk]]["übersehen"] += 1
                markierungen.append((wb, "#f58231", f"übersehen {namen[wk]}"))
                fehler += 1
        pro_bild.append((fehler, r.path, markierungen))
    tab = pd.DataFrame(zaehler).T
    tab["nicht gefunden"] = (1 - tab.richtig / (tab.richtig + tab.verwechselt + tab["übersehen"]).clip(lower=1)).round(2)
    display(tab)
    schlimmste = [b for b in sorted(pro_bild, key=lambda b: -b[0]) if b[0] > 0][:zeigen]
    if not schlimmste:
        print("🎉 Keine Fehler in dieser Teilmenge!")
        return tab
    print(f"Die {len(schlimmste)} Bilder mit den meisten Fehlern  (grün = richtig · violett = verwechselt · "
          "rot = Fehlalarm · orange = übersehen):")
    spalten = 3
    zeilen = -(-len(schlimmste) // spalten)
    fig, ax = plt.subplots(zeilen, spalten, figsize=(17, 5 * zeilen))
    for a in np.atleast_1d(ax).flat:
        a.axis("off")
    for a, (n, pfad, mark) in zip(np.atleast_1d(ax).flat, schlimmste):
        im = Image.open(pfad).convert("RGB")
        d = ImageDraw.Draw(im)
        schrift = ImageFont.truetype(SCHRIFT, max(16, im.width // 45))
        for box, farbe, text in mark:
            d.rectangle(box, outline=farbe, width=max(3, im.width // 250))
            tb = d.textbbox((0, 0), text, font=schrift)
            y = box[1] - (tb[3] - tb[1]) - 8 if box[1] > (tb[3] - tb[1]) + 8 else box[3] + 2
            d.rectangle([box[0], y, box[0] + tb[2] - tb[0] + 8, y + tb[3] - tb[1] + 6], fill=farbe)
            d.text((box[0] + 4, y + 1), text, fill="white", font=schrift)
        a.imshow(im)
        a.set_title(f"{Path(pfad).name}  ({n} Fehler)", fontsize=9)
    plt.tight_layout()
    plt.show()
    return tab

In [ ]:
fehler_val = fehleranalyse(BEST, DATA_YAML, "val", conf=BESTE_CONF)

### ✏️ Diagnose-Blatt (Doppelklick zum Ausfüllen)

Daraus entsteht dein Aufnahmeplan für das Nachtraining.

| Klasse | häufigste Fehlerart | typische Situation (Licht, Hintergrund, Größe, Lage) | Label falsch? | geplante neue Fotos (Anzahl, welche) |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |

---
## 7 · Modell sichern

`best.pt` wird unter einem eindeutigen Namen in **`YOLO_Projekt/modelle/`** in Drive abgelegt, zusammen mit einer
Infodatei (Datensatz, Kennzahlen, bester Schwellenwert). Die heruntergeladene Datei landet im Ordner *Downloads*.
Verschiebe sie in deinen Projektordner und teste sie dort mit der Webcam. Den passenden Befehl gibt die Zelle aus.

In [ ]:
ZIEL_MODELL = MODELLE / f"{LAUF}_best.pt"
shutil.copy2(BEST, ZIEL_MODELL)
(MODELLE / f"{LAUF}_info.json").write_text(json.dumps({
    "lauf": LAUF, "datensatz": DATENSATZ, "startmodell": str(MODELL), "bildgroesse": BILDGROESSE,
    "zusatz": ZUSATZ, "beste_conf": round(BESTE_CONF, 3), "val_mAP50": round(float(val.box.map50), 4),
    "val_mAP50_95": round(float(val.box.map), 4)}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"💾 gespeichert: {ZIEL_MODELL}")
print("\nTesten auf dem Laptop (im Projektordner):")
print(f"   uv run yolo predict model={ZIEL_MODELL.name} source=0 show=True conf={BESTE_CONF:.2f}")

HERUNTERLADEN = True
if IN_COLAB and HERUNTERLADEN:
    from google.colab import files
    files.download(str(ZIEL_MODELL))

---
## 8 · Ausprobieren mit eigenen Fotos

Lade ein paar **neue** Fotos hoch, am besten aus der echten Einsatzsituation. Wenn du nichts hochlädst, werden
drei Validierungsbilder verwendet. Die Testbilder bleiben für den Abschlusstest unberührt.

In [ ]:
if IN_COLAB:
    from google.colab import files
    hochgeladen = files.upload()          # Dateiauswahl erscheint unter der Zelle
    fotos = list(hochgeladen.keys())
else:
    fotos = [str(p) for p in sorted(Path("eigene_bilder").glob("*.jpg"))] if Path("eigene_bilder").exists() else []

if not fotos:  # Ersatz: Validierungsbilder
    fotos = [str(p) for p in sorted((DATA_YAML.parent / "images" / "val").glob("*.jpg"))[:3]]
for r in modell.predict(fotos, conf=BESTE_CONF, imgsz=BILDGROESSE, verbose=False):
    gefunden = ", ".join(modell.names[int(c)] for c in r.boxes.cls) or "nichts erkannt"
    plt.figure(figsize=(9, 6))
    plt.imshow(r.plot()[..., ::-1])
    plt.axis("off")
    plt.title(f"{Path(r.path).name}: {gefunden}")
    plt.show()

---
## 9 · Vergleich mit dem vorherigen Lauf *(nur beim Nachtraining)*

Ist `VERGLEICH_MIT` gesetzt, werden **beide** Modelle auf **denselben** Bildern geprüft: den val-Bildern des
**aktuellen** Datensatzes. Wer das alte Modell auf alten und das neue auf neuen Bildern misst, vergleicht Äpfel mit Birnen.

Kam eine **neue Klasse** dazu, kann das alte Modell sie nicht kennen. Verglichen werden dann nur die **gemeinsamen** Klassen.

⚠️ **Kleine Unterschiede sind Zufall.** Mit 20–60 val-Bildern schwankt mAP50-95 schon bei zwei identischen Trainings
um 0,02–0,05. Erst ab etwa **0,03** lohnt es sich, von „besser“ oder „schlechter“ zu sprechen.
Ausführlich: [Nachtraining](https://rdf-sul.github.io/YOLO_Training/projekt/nachtraining/)

In [ ]:
def messen(pfad, data_yaml):
    m = YOLO(str(pfad))
    r = m.val(data=str(data_yaml), split="val", imgsz=BILDGROESSE, plots=False, verbose=False,
              project=str(DATEN_LOKAL / "auswertung"), name="vergleich", exist_ok=True)
    return {r.names[int(c)]: r.box.maps[int(c)] for c in r.box.ap_class_index}, m.names   # m.names = Klassen des Modells

if VERGLEICH_MIT:
    alt_pfad = MODELLE / f"{VERGLEICH_MIT}_best.pt"
    if not alt_pfad.exists():
        alt_pfad = LAEUFE / VERGLEICH_MIT / "weights" / "best.pt"
    alt, alt_namen = messen(alt_pfad, DATA_YAML)
    neu, neu_namen = messen(BEST, DATA_YAML)
    gemeinsam = [k for k in neu if k in alt and k in alt_namen.values()]
    nur_neu = [k for k in neu if k not in alt_namen.values()]
    vergleich = pd.DataFrame({f"alt ({VERGLEICH_MIT})": {k: alt[k] for k in gemeinsam},
                              f"neu ({LAUF})": {k: neu[k] for k in gemeinsam}}).round(3)
    vergleich.loc["Mittelwert (gemeinsame Klassen)"] = vergleich.mean().round(3)
    vergleich["Differenz"] = (vergleich.iloc[:, 1] - vergleich.iloc[:, 0]).round(3)
    print("mAP50-95 pro Klasse auf den val-Bildern von", DATENSATZ)
    display(vergleich.style.background_gradient(subset=["Differenz"], cmap="RdYlGn", vmin=-.1, vmax=.1))
    vergleich.iloc[:-1, :2].plot.barh(figsize=(8, 0.5 * len(vergleich) + 1)); plt.xlim(0, 1); plt.show()
    if nur_neu:
        print(f"ℹ️  Neue Klasse(n) {nur_neu}: nur das neue Modell kennt sie → mAP50-95 "
              + ", ".join(f"{k} = {neu[k]:.3f}" for k in nur_neu))
    d = vergleich.loc["Mittelwert (gemeinsame Klassen)", "Differenz"]
    print("➡️ " + (f"Das neue Modell ist deutlich besser (+{d:.3f}) → {LAUF}_best.pt verwenden." if d >= 0.03 else
                   f"Das neue Modell ist deutlich schlechter ({d:.3f}) → Labels der neuen Bilder prüfen!" if d <= -0.03 else
                   f"Kein deutlicher Unterschied ({d:+.3f}) – das kann Zufall sein. Prüfe mit der Fehleranalyse, "
                   "ob die neuen Bilder die Schwäche treffen."))
else:
    print("Kein Vergleich (VERGLEICH_MIT ist leer).")

---
## 10 · Abschlusstest mit den Testbildern *(nur ganz am Ende)*

Die Testbilder hat das Modell **nie** gesehen, und **du** hast sie bisher auch nicht zum Entscheiden benutzt.
Sie liefern deshalb die ehrlichste Bewertung, aber nur, wenn du sie **ein einziges Mal** verwendest.
Wer nach dem Test noch etwas ändert und neu trainiert, passt sein Modell unbewusst an den Test an, und er ist nichts mehr wert.

👉 Setze oben `ABSCHLUSSTEST = True` erst, wenn du dich für dein **endgültiges** Modell entschieden hast,
und führe dann diesen Lauf noch einmal komplett aus. Ist das Ergebnis deutlich schlechter als auf *val*,
war *val* zu klein oder zu einfach.

In [ ]:
if not ABSCHLUSSTEST:
    print("⏸️  Abschlusstest übersprungen (ABSCHLUSSTEST = False).")
elif "test" not in info["anzahl"]:
    print("Dieser Datensatz hat keine Testbilder.")
else:
    test = modell.val(data=str(DATA_YAML), split="test", imgsz=BILDGROESSE, plots=False,
                      project=str(DATEN_LOKAL / "auswertung"), name=f"{LAUF}_test", exist_ok=True, verbose=False)
    ergebnis = pd.DataFrame({"val": [val.box.mp, val.box.mr, val.box.map50, val.box.map],
                             "test": [test.box.mp, test.box.mr, test.box.map50, test.box.map]},
                            index=["Precision", "Recall", "mAP50", "mAP50-95"]).round(3)
    display(ergebnis)
    (MODELLE / f"{LAUF}_abschlusstest.json").write_text(ergebnis.to_json(indent=2), encoding="utf-8")
    print(f"💾 Ergebnis gespeichert: {MODELLE / f'{LAUF}_abschlusstest.json'}")

✅ **Runde fertig!** Trage Datensatz-Version, Laufname, mAP50-95 auf val, schwächste Klasse und deine Änderung
ins **Versions-Logbuch** ein. Für die nächste Runde: Einstellungen oben ändern → **Laufzeit → Alle ausführen**.